# Libraries management

In [ ]:
import oracledb as oracledb       # python-oracledb: same API as cx_Oracle, no sys.modules alias needed
import numpy as np
import pandas as pd
import win32com.client as win32
import gc as gc
from time import sleep, perf_counter
import sys as sys
import os as os
from datetime import datetime as datetime
# import matplotlib.pyplot as plt
# from minio import Minio
# import smtplib
## Check python version
import platform

from dotenv import load_dotenv
load_dotenv()

In [ ]:
# print(sys.version)
# print(platform.python_version())
# print(oracledb.__version__, pd.__version__)

# Config management

In [ ]:
instant_client_path = r"D:\oracle\instantclient_19_29"


def log(message):
    """Prints with a timestamp so the batch log (Automation_Log*.txt) shows how long each step takes."""
    print(f"[{datetime.now():%Y-%m-%d %H:%M:%S}] {message}", flush=True)


def init_oracle_client_once(lib_dir):
    """Loads the Oracle Instant Client (Thick mode) once per kernel, so this cell is safe to re-run."""
    if hasattr(oracledb, "is_thin_mode") and not oracledb.is_thin_mode():
        log("Oracle Client already initialised in this kernel.")
        return
    if not os.path.isdir(lib_dir):
        raise FileNotFoundError(f"Oracle Instant Client folder not found: {lib_dir}")
    oracledb.init_oracle_client(lib_dir=lib_dir)
    log("Initialization successful.")


init_oracle_client_once(instant_client_path)

# Declaration of variables

In [ ]:
ip = os.getenv("DWH_HOST")
port = os.getenv("DWH_PORT")
service_name = os.getenv("DWH_NAME")
username = os.getenv("DWH_USER")
password = os.getenv("DWH_PASSWORD")

# --- Fail fast (and never print the password) if the .env file is missing or incomplete ---
required_env = ["DWH_HOST", "DWH_PORT", "DWH_NAME", "DWH_USER", "DWH_PASSWORD"]
missing_env = [key for key in required_env if not os.getenv(key)]
if missing_env:
    raise EnvironmentError(f"Missing variables in .env: {', '.join(missing_env)}")

sql_script = """
-- Tet windows are written once, as ANSI DATE literals (independent of the session NLS_DATE_FORMAT).
WITH base AS (
    SELECT
        sale_date,
        supplier_code,
        supplier_name,
        dimension_group,
        dimension,
        net_sales,
        ord_cnt,
        byr_cnt,
        margin,
        CASE
            WHEN sale_date BETWEEN DATE '2024-12-15' AND DATE '2025-02-13' THEN 'Tet 2025'
            WHEN sale_date BETWEEN DATE '2026-01-03' AND DATE '2026-03-04' THEN 'Tet 2026'
            ELSE 'Other Period'
        END AS campaign_period
    FROM
        crv_data.loutruong_supplier_perf_di
    WHERE
        1 = 1
        AND (
            sale_date BETWEEN DATE '2024-12-15' AND DATE '2025-02-13'
            OR sale_date BETWEEN DATE '2026-01-03' AND DATE '2026-03-04'
        )
        AND supplier_code IN (
            SELECT
                supplier_code
            FROM
                omni_digimgr.loutruong_dim_supplier
        )
)
SELECT
    base.*,
    DENSE_RANK() OVER (
        PARTITION BY campaign_period
        ORDER BY TRUNC(sale_date)
    ) AS day_number
FROM
    base
ORDER BY
    sale_date ASC,
    supplier_code ASC,
    dimension_group ASC,
    dimension ASC
"""
sheet_path = r"D:\OneDrive - Central Group\Stella's files - 1. HAND OVER\03. REPORT DAILY\09_supplier_tracker\Supplier_Performance_Tracker.xlsx"
sheet_name = 'perf_raw_di_tet'

# --- Tuning knobs (change here, not in the logic below) ---
fetch_arraysize = 10000        # rows per round trip when fetching from Oracle (default 100 = slow on 400k+ rows)
allow_empty_result = False     # False = stop before Excel is touched when the query returns 0 rows (upstream not loaded yet)
open_retries = 5               # attempts to open the workbook while OneDrive / another process still holds it
open_retry_wait = 15           # seconds between open attempts
load_wait = 3                  # seconds to let Excel settle after the workbook is opened
paste_chunk_rows = 50000       # rows per COM call, keeps memory flat on very large pastes
strict_header_check = False    # True = stop the run when the Excel header row differs from the query columns
refresh_after_paste = True     # refresh queries / pivots once the new raw data is in the sheet
refresh_timeout = 600          # max seconds to wait for background queries to finish
refresh_settle_wait = 2        # seconds to let Excel settle after the refresh

# 3. Connection controller

In [ ]:
def fetch_dataframe(sql, arraysize):
    """Runs the query on the DWH and returns a DataFrame. Connection and cursor are always closed, even on error."""
    started = perf_counter()
    with oracledb.connect(user=username, password=password,
                          host=ip, port=int(port), service_name=service_name) as connection:
        with connection.cursor() as cursor:
            cursor.arraysize = arraysize          # fewer round trips for big result sets
            cursor.prefetchrows = arraysize + 1
            cursor.execute(sql)
            columns = [col[0] for col in cursor.description]
            rows = cursor.fetchall()
    df = pd.DataFrame(rows, columns=columns)
    log(f"Fetched {len(df):,} rows x {len(df.columns)} columns in {perf_counter() - started:,.1f}s.")
    return df

In [ ]:
try:
    df = fetch_dataframe(sql_script, fetch_arraysize)
    if df.empty and not allow_empty_result:
        raise ValueError("Query returned 0 rows - refusing to wipe the sheet. Check the DWH load, or set allow_empty_result = True.")
except Exception as e:
    log(f"SCRIPT_RESULT: FAILED; ERROR: {type(e).__name__}: {e}")
    raise

In [ ]:
# print(df.head(10).to_string())

# print(df.dtypes)

# Main logic: Open > Delete > Write > Refresh > Save > Close

In [ ]:
# --- Helper functions: each one does exactly one step of Open > Delete > Write > Refresh > Save > Close ---
def column_to_letter(col_num):
    """Converts a column number (e.g., 1) into its Excel letter (e.g., 'A')."""
    string = ""
    while col_num > 0:
        col_num, remainder = divmod(col_num - 1, 26)
        string = chr(65 + remainder) + string
    return string


def prepare_data(df):
    """Makes the DataFrame safe for COM: dates as 'YYYY-MM-DD' text, NULL/NaN as blank cells,
    native Python types only, and text starting with '=' escaped so Excel never treats it as a formula."""
    df_clean = df.copy()
    for col in df_clean.select_dtypes(include=["datetime", "datetimetz"]).columns:
        df_clean[col] = df_clean[col].dt.strftime("%Y-%m-%d")
    df_clean = df_clean.astype(object).where(df_clean.notna(), "")
    for col in df_clean.columns:
        is_formula = df_clean[col].map(lambda v: isinstance(v, str) and v.startswith("="))
        if is_formula.any():
            df_clean.loc[is_formula, col] = "'" + df_clean.loc[is_formula, col]
    return df_clean


def start_excel():
    """Starts a dedicated, hidden Excel instance (DispatchEx = never hijacks an Excel the user has open)."""
    xl = win32.DispatchEx("Excel.Application")
    xl.Visible = False          # Open in background. should be True for debugging
    xl.DisplayAlerts = False    # To avoid any pop ups
    xl.ScreenUpdating = False
    return xl


def open_workbook(xl, path, retries, wait_seconds):
    """Opens the workbook, retrying while OneDrive / another process still holds the file
    (the 'Microsoft Excel cannot access the file' error). Refuses to continue on a read-only copy."""
    if not os.path.isfile(path):
        raise FileNotFoundError(f"Workbook not found: {path}")
    retries = max(1, int(retries))
    for attempt in range(1, retries + 1):
        try:
            wb = xl.Workbooks.Open(path)
            break
        except Exception as e:
            if attempt == retries:
                raise
            log(f"!! Open attempt {attempt}/{retries} failed ({type(e).__name__}). Retrying in {wait_seconds}s.")
            sleep(wait_seconds)
    if wb.ReadOnly:
        wb.Close(SaveChanges=False)
        raise PermissionError(f"Workbook opened READ-ONLY (locked by another user or process): {path}")
    try:
        wb.AutoSaveOn = False   # never let OneDrive AutoSave persist a half-finished run
    except Exception:
        pass                    # older Excel / non-cloud file: property not available, nothing to switch off
    log(f"Opened '{wb.Name}'.")
    return wb


def get_sheet(wb, name):
    """Grabs the specific worksheet object."""
    try:
        return wb.Sheets(name)
    except Exception as e:
        raise LookupError(f"Could not find sheet '{name}' in '{wb.Name}'. Please check the name.") from e


def check_header(ws, columns, strict):
    """Compares the sheet header (row 1) with the query columns so data never lands under the wrong header."""
    header = ws.Range(ws.Cells(1, 1), ws.Cells(1, len(columns))).Value
    if not isinstance(header, (tuple, list)):      # a 1-column range comes back as a scalar
        header = ((header,),)
    excel_header = [str(h).strip().upper() if h is not None else "" for h in header[0]]
    query_header = [str(c).strip().upper() for c in columns]
    if excel_header != query_header:
        message = (f"Header mismatch in sheet '{ws.Name}'.\n"
                   f"    Excel : {excel_header}\n"
                   f"    Query : {query_header}")
        if strict:
            raise ValueError(message)
        log(f"!! {message}")


def clear_sheet(ws):
    """Clears everything below the header row so no stale rows survive a shorter refresh."""
    used = ws.UsedRange
    last_row = used.Row + used.Rows.Count - 1
    last_col = used.Column + used.Columns.Count - 1
    if last_row < 2:
        log(f"Sheet '{ws.Name}' is empty or only contains a header row. No data was cleared.")
        return
    ws.Range(ws.Cells(2, 1), ws.Cells(last_row, last_col)).ClearContents()
    log(f"Cleared data from Row 2 to {last_row} (Columns A to {column_to_letter(last_col)}) in sheet '{ws.Name}'.")


def paste_dataframe(ws, df_clean, chunk_rows):
    """Pastes the DataFrame from A2 downward in chunks. Returns the number of rows pasted."""
    num_rows, num_cols = df_clean.shape
    if num_rows == 0:
        log("DataFrame is empty. Nothing to paste.")
        return 0
    values = df_clean.values.tolist()
    for start in range(0, num_rows, chunk_rows):
        chunk = values[start:start + chunk_rows]
        first_row = 2 + start
        last_row = first_row + len(chunk) - 1
        ws.Range(ws.Cells(first_row, 1), ws.Cells(last_row, num_cols)).Value = chunk
        log(f"    pasted rows {first_row:,} - {last_row:,}")
    log(f"Successfully pasted {num_rows} rows into sheet '{ws.Name}', starting at A2.")
    return num_rows


def still_refreshing(wb):
    """True while any OLEDB / ODBC connection reports it is still refreshing. False when Excel cannot tell us."""
    try:
        for cn in wb.Connections:
            for kind in ("OLEDBConnection", "ODBCConnection"):
                try:
                    if getattr(cn, kind).Refreshing:
                        return True
                except Exception:
                    pass        # this connection type has no such property
    except Exception:
        pass                    # no Connections collection - nothing to wait for
    return False


def refresh_workbook(xl, wb, timeout_seconds, settle_seconds):
    """Refreshes all queries / pivots and waits until background queries are really finished,
    so Save() never happens in the middle of a refresh."""
    wb.RefreshAll()
    xl.CalculateUntilAsyncQueriesDone()
    deadline = perf_counter() + timeout_seconds
    while still_refreshing(wb):
        if perf_counter() >= deadline:
            log(f"!! Background queries still running after {timeout_seconds}s. Continuing anyway.")
            break
        sleep(2)
    sleep(settle_seconds)
    log("Done Refresh")


def close_excel(xl, wb):
    """Closes the workbook WITHOUT saving (Save() is called explicitly on success only) and quits Excel.
    Called from a finally block, so a failed run can never leave a hidden EXCEL.EXE holding the file.
    No pythoncom.CoUninitialize() here: pywin32 initialises COM on import and releases it at exit,
    and calling it manually breaks any Excel call made later in the same kernel."""
    if wb is not None:
        try:
            wb.Close(SaveChanges=False)
        except Exception as e:
            log(f"!! Could not close the workbook cleanly: {e}")
    if xl is not None:
        try:
            xl.Quit()
        except Exception as e:
            log(f"!! Could not quit Excel cleanly: {e}")
    log("Excel closed.")

In [ ]:
xl = wb = ws = None
rows_processed = 0
started = perf_counter()

try:
    # 1. Open
    xl = start_excel()
    wb = open_workbook(xl, sheet_path, open_retries, open_retry_wait)
    sleep(load_wait)
    ws = get_sheet(wb, sheet_name)
    check_header(ws, df.columns, strict_header_check)

    # 2. Delete
    clear_sheet(ws)

    # 3. Write
    rows_processed = paste_dataframe(ws, prepare_data(df), paste_chunk_rows)

    # 4. Refresh (after the paste, so pivots / queries see the new raw data)
    if refresh_after_paste:
        refresh_workbook(xl, wb, refresh_timeout, refresh_settle_wait)

    # 5. Save - the only place the workbook is ever saved
    wb.Save()
    log(f"Saved '{wb.Name}'.")
    log(f"SCRIPT_RESULT: SUCCESS; ROWS_PROCESSED: {rows_processed}; ELAPSED_SECONDS: {perf_counter() - started:,.0f}")

except Exception as e:
    log(f"SCRIPT_RESULT: FAILED; ERROR: {type(e).__name__}: {e}")
    raise

finally:
    # 6. Close - runs on success AND on failure
    close_excel(xl, wb)
    ws = wb = xl = None     # drop every COM reference so EXCEL.EXE can actually exit
    gc.collect()